# Graph Memory: Reasoning Over Relationships, Not Just Similarity

Semantic memory (Demo 02) retrieves by *similarity* but can't reason over *relationships*.
A multi-hop question needs to find an entry point by similarity, then **traverse the graph**.

**The question this demo answers:** *"Who do I know that's connected to flights to Spain?"*

Based on research:
- [GAAMA: Graph Augmented Associative Memory for Agents](https://arxiv.org/abs/2603.27910), Paul et al., 2026
- [MAGMA: A Multi-Graph based Agentic Memory Architecture for AI Agents](https://arxiv.org/abs/2601.03236), Jiang et al., 2026
- [GRAVITY: Architecture-Agnostic Structured Anchoring for Long-Horizon Conversational Memory](https://arxiv.org/abs/2605.01688), Sun et al., 2026

Uses [Strands Agents](https://github.com/strands-agents/sdk-python) for the harness and
[Neo4j](https://neo4j.com/) + [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/) for graph memory.

## Prerequisites

1. A running **Neo4j** (Desktop, Docker, or Aura).
2. Environment variables in a `.env` file (copy from `.env.example`):
   `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

The demo uses OpenAI by default; swap the model/embeddings for Amazon Bedrock in production (Demo 07).

## Install dependencies

Run this once (or install from a terminal with `uv venv && uv pip install -r requirements.txt`).

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configure your model provider

The demo runs with **OpenAI** by default, but you can use **Amazon Bedrock**, **Anthropic**, or any provider available in the Strands configuration. See [supported model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

- **OpenAI (default):** set `OPENAI_API_KEY` below or in a `.env` file. Get one at https://platform.openai.com/api-keys
- **Amazon Bedrock:** no OpenAI key needed, uses your AWS credentials (`aws configure`, with model access enabled in your region). In Test 3's cell (where the chat model is created), comment the `OpenAIModel` lines and uncomment the Bedrock block.

In [2]:
import os

# python-dotenv loads OPENAI_API_KEY and the NEO4J_* connection values from .env,
# so credentials never live inside the notebook.
from dotenv import load_dotenv
load_dotenv()

USING_OPENAI = True  # set False if you switch the chat model to Bedrock in the model cell below
if USING_OPENAI:
    assert os.getenv('OPENAI_API_KEY'), (
        'OPENAI_API_KEY not set: needed for the chat model AND the embeddings in this demo. '
        'Get yours at https://platform.openai.com/api-keys, or switch the chat model to Bedrock below.'
    )
assert os.getenv('NEO4J_PASSWORD'), 'Set NEO4J_* values in your .env'
print('Provider configured')

Provider configured


## Setup: build the graph memory

`graph_memory.build()` connects to Neo4j, ensures an isolated database (and Cypher 25 where needed),
seeds the known graph with real embeddings, and creates the native vector index.

In [ ]:
import graph_memory as gm

driver, db, embedder = gm.build()
QUESTION = "Who do I know that's connected to flights to Spain?"
print('Question:', QUESTION)

## The seeded graph

What the agent learned across sessions, stored as connected nodes:

```
(Maya Torres) ─WORKS_AT→ (Iberia) ─MEMBER_OF→ (Oneworld)
                              │
                         FLIES_TO
                              ▼
                          (Madrid) ─IN_COUNTRY→ (Spain)
```

The answer to the question, **Maya Torres**, is never stated; you can only reach it by following edges.

In [ ]:
import os
import travel_tools as tt
from strands import Agent
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

os.environ['OTEL_SDK_DISABLED'] = 'true'

MODEL = OpenAIModel(model_id='gpt-4o-mini')
tt.init_memory(driver=driver, db=db, embedder=embedder)

# To run on Amazon Bedrock instead (no OpenAI key), comment the two lines above and uncomment:
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

print('Model and tools ready.')

---
## Test 1: Agent with semantic recall only

The agent has only `recall_semantic`: pure vector similarity, no traversal.
It surfaces related pieces (Iberia, Madrid, Spain) but cannot connect them to a person.

In [ ]:
agent_semantic = Agent(
    model=MODEL,
    system_prompt="You are a personal travel assistant with access to the user's travel memory. Be concise.",
    tools=[tt.recall_semantic],
    callback_handler=None,
)

print(f'Question: {QUESTION}\n')
resp = agent_semantic(QUESTION)
print('Agent:', resp.message['content'][0]['text'].strip())
print('\nRecovers Maya Torres?', 'Maya Torres' in resp.message['content'][0]['text'])

---
## Test 2: Agent with graph recall

The agent has only `recall_graph`: vector similarity finds an entry node, then Cypher
traversal walks the relationships back to the person and returns the full chain.

In [ ]:
agent_graph = Agent(
    model=MODEL,
    system_prompt="You are a personal travel assistant with access to the user's travel memory. Be concise.",
    tools=[tt.recall_graph],
    callback_handler=None,
)

print(f'Question: {QUESTION}\n')
resp = agent_graph(QUESTION)
print('Agent:', resp.message['content'][0]['text'].strip())
print('\nRecovers Maya Torres?', 'Maya Torres' in resp.message['content'][0]['text'])

---
## Test 3: A full Strands agent with graph memory

The agent uses `recall_graph` to answer, and `remember_fact` to write a new fact back into the graph:
the harness is just tools + state.

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="You are a personal travel assistant with access to the user's travel memory. Always store new facts the user shares. Be concise.",
    tools=[tt.search_flights, tt.book_flight, tt.best_time_to_visit,
           tt.recall_graph, tt.recall_semantic, tt.remember_fact],
    callback_handler=None,
)

resp = agent(QUESTION)
print('Agent:', resp.message['content'][0]['text'].strip())

In [7]:
resp = agent('By the way, remember that Maya Torres works at Iberia. She is my contact there.')
print('Agent:', resp.message['content'][0]['text'].strip())
print('\nFacts logged to agent.state:', agent.state.get('remembered_facts'))

Agent: Got it! I've noted that Maya Torres works at Iberia and is your contact there.

Facts logged to agent.state: [{'subject': 'Maya Torres', 'relation': 'WORKS_AT', 'object': 'Iberia'}]


---
## Test 4: Deterministic scorecard

Four multi-hop questions, checked against the known graph (no LLM judge). Reproducible.

In [ ]:
SCORECARD = [
    ("Who do I know that's connected to flights to Spain?", 'Maya Torres'),
    ('Who do I know connected to an airline that flies to Madrid?', 'Maya Torres'),
    ('Who works at the Oneworld airline I know?', 'Maya Torres'),
    ('Which person is linked to airlines in Spain?', 'Maya Torres'),
]

semantic_retriever = gm.make_semantic_retriever(driver, db, embedder)
graph_retriever = gm.make_graph_retriever(driver, db, embedder)

semantic_hits = graph_hits = 0
print(f"{'Question':<52}{'semantic':>10}{'graph':>7}")
for q, target in SCORECARD:
    s = any(target in it.content for it in semantic_retriever.search(query_text=q, top_k=3).items)
    g = any(target in it.content for it in graph_retriever.search(query_text=q, top_k=3).items)
    semantic_hits += s; graph_hits += g
    print(f"{q[:52]:<52}{('OK' if s else '-'):>10}{('OK' if g else '-'):>7}")

print(f'\nCorrect: semantic: {semantic_hits}/{len(SCORECARD)} | graph: {graph_hits}/{len(SCORECARD)}')

---
## Summary

| Test | Strategy | Multi-hop answer? |
|------|----------|-------------------|
| Test 1: Agent: semantic recall | `recall_semantic`, pure vector similarity | No: finds pieces, can't connect them |
| Test 2: Agent: graph recall | `recall_graph`, similarity + traversal | Yes: returns the full chain |
| Test 3: Agent: graph + write | Both tools + `remember_fact` | Yes: and writes new facts back |

**Key insight:** similarity finds related pieces; only traversal connects them. Graph memory answers
multi-hop questions that flat/semantic memory cannot, and with Strands, plugging in the graph store
is just tools + state.

Both retrievers received the **same facts** and shared the **same vector index**. The graph wins
structurally, not because it was handed the answer.

In [9]:
driver.close()
print('Done.')

Done.
